In [0]:
spark.sql(
    "DROP TABLE IF EXISTS workspace.gold.dim_alert"
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_alert (
    alert_key BIGINT,
    alert_level STRING,
    alert_description STRING
)
""")

In [0]:
from pyspark.sql import functions as F

alert = (
    spark.table(
        "workspace.silver.usgs_earthquakes"
    )
    .select("alert")
    .fillna({"alert": "No alert"})
    .dropDuplicates()
    .withColumn(
        "alert_description",
        F.when(
            F.col("alert") == "green",
            "Alerta verde"
        )
        .when(
            F.col("alert") == "yellow",
            "Alerta amarilla"
        )
        .when(
            F.col("alert") == "orange",
            "Alerta naranja"
        )
        .when(
            F.col("alert") == "red",
            "Alerta roja"
        )
        .otherwise(
            "Sin alerta"
        )
    )
    .withColumn(
        "alert_key",
        F.xxhash64("alert")
    )
)

alert = alert.withColumnRenamed("alert", "alert_level").select(
    "alert_key",
    "alert_level",
    "alert_description"
)

In [0]:
alert.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.gold.dim_alert")